[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/08_heap.ipynb)

*DSA Interview Handbook · Notebook 8 of 14*

# Heap / Priority Queue

Always know the smallest (or largest) item in O(1) and update it in O(log n) — top-k, merging, scheduling.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A heap is a hospital triage desk: the most urgent patient is always on top, without sorting everyone.

**What interviewers really test here:** Top-k patterns, two-heap medians and heap vs sort trade-offs.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. What a priority queue is *(Basics)*
- [ ] 2. How a heap is stored *(Basics)*
- [ ] 3. Using heapq in Python *(Core)*
- [ ] 4. Top-k with a heap of size k *(Core)*
- [ ] 5. Merging k sorted things *(Core)*
- [ ] 6. Two heaps for a running median *(Advanced)*
- [ ] 7. Greedy scheduling with a heap *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **heap** is a **complete binary tree** that satisfies the **heap property**:




- **Min-heap**: every parent's key is ≤ its children's keys → the minimum is at the root.
- **Max-heap**: every parent's key is ≥ its children's keys → the maximum is at the root.



A **priority queue** is an ADT where each element has a priority and the element with the highest priority is removed first. A binary heap is the standard way to implement it.



Note: a heap is **not sorted**. Siblings have no order between them; only the parent–child relationship is guaranteed.

### Array representation

Because a heap is a complete tree, it is stored in an array with no gaps and no pointers. For a node at index i (0-based):



| Relation | Index |
| --- | --- |
| left child | 2i + 1 |
| right child | 2i + 2 |
| parent | (i − 1) // 2 |
| leaves | indices n // 2 … n − 1 |



Height of a heap with n nodes is ⌊log₂ n⌋.

### Operations

| Operation | Steps | Time |
| --- | --- | --- |
| **Insert** (push) | append at the end, then **heapify-up** (sift up): swap with the parent while it violates the heap property | O(log n) |
| **Extract-min/max** (pop) | save the root, move the last element to the root, then **heapify-down** (sift down): swap with the smaller (min-heap) child while it violates the property | O(log n) |
| Peek | read index 0 | O(1) |
| **Build heap** (heapify) | sift down every non-leaf from index n//2 − 1 down to 0 | **O(n)** |
| Decrease / increase key | change the value, sift up or down | O(log n) (if index known) |
| Search arbitrary value | scan | O(n) |
| Delete arbitrary element | find, swap with last, sift | O(n) find + O(log n) |



#### Why build-heap is O(n), not O(n log n)



Most nodes are near the bottom and sift down only a little. About n/2 nodes are leaves (0 work), n/4 sift down at most 1 level, n/8 at most 2 levels, and so on. The total Σ (n / 2ᵏ⁺¹) · k is bounded by n, so building a heap from an array is linear.

### Heap sort

1. Build a max-heap from the array — O(n).
2. Repeat n times: swap the root (max) with the last element of the heap, shrink the heap by one, and sift down the new root — O(log n) each.



Total **O(n log n)** in all cases, **O(1)** extra space (in place), but **not stable** and usually slower than quicksort in practice due to poor cache use.

### Python's heapq

| Function | Does | Time |
| --- | --- | --- |
| `heappush(h, x)` | insert | O(log n) |
| `heappop(h)` | remove smallest | O(log n) |
| `heapify(lst)` | build in place | O(n) |
| `heappushpop(h, x)` | push then pop (faster than both) | O(log n) |
| `heapreplace(h, x)` | pop then push | O(log n) |
| `nlargest(k, it)` / `nsmallest` | top k | O(n log k) |



`heapq` implements a **min-heap** only. Max-heap: store negated values. Custom priority: store tuples `(priority, counter, item)`; the counter breaks ties and avoids comparing items.

### Comparison and applications

| Structure | insert | get min | remove min | search |
| --- | --- | --- | --- | --- |
| Unsorted array | O(1) | O(n) | O(n) | O(n) |
| Sorted array | O(n) | O(1) | O(1) (from the end) | O(log n) |
| **Binary heap** | O(log n) | O(1) | O(log n) | O(n) |
| Balanced BST | O(log n) | O(log n) | O(log n) | O(log n) |



#### Applications




- Operating system and job scheduling by priority.
- **Dijkstra's** and **Prim's** algorithms; A\* search.
- Event-driven simulation (process the earliest event next).
- Top-k queries, leaderboards, streaming medians.
- Merging k sorted files (external sorting); Huffman coding.
- Load balancing (send work to the least-loaded server).

---
# Part 2: Lessons, step by step

## Lesson 1: What a priority queue is

*Level: Basics*

A normal queue serves people in arrival order. A **priority queue** always serves the **most important** item next — like an emergency room. You need two fast operations: add an item, and remove the smallest (or largest).



| Option | add | remove min |
| --- | --- | --- |
| unsorted list | O(1) | O(n) — search |
| sorted list | O(n) — shift | O(1) |
| **binary heap** | **O(log n)** | **O(log n)** |



A heap is the best of both: never fully sorted, but always knows its smallest item.

## Lesson 2: How a heap is stored

*Level: Basics*

A **min-heap** is a binary tree where every parent ≤ its children, filled level by level from left to right. Because it's always that compact shape, it fits in a plain list with no pointers:



```text
index i → children 2i + 1 and 2i + 2, parent (i − 1) // 2

heap = [1, 3, 2, 7, 4]          1
                              /   \
                             3     2
                            / \
                           7   4
```




- **Push**: put the new value at the end, then **sift up** — swap with the parent while it is smaller.
- **Pop**: take `heap[0]`, move the last value to the root, then **sift down** — swap with the smaller child while it is bigger.



The tree has about log₂ n levels, so each sift is O(log n). Building a heap from n items with `heapify` is O(n).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def sift_lesson():
    heap = [1, 3, 2, 7, 4]
    heap.append(0)
    i = len(heap) - 1
    while i > 0 and heap[(i - 1) // 2] > heap[i]:
        p = (i - 1) // 2
        heap[i], heap[p] = heap[p], heap[i]
        i = p
    top = heap[0]
    heap[0] = heap.pop()
    i = 0
    while True:
        l, r, small = 2 * i + 1, 2 * i + 2, i
        if l < len(heap) and heap[l] < heap[small]:
            small = l
        if r < len(heap) and heap[r] < heap[small]:
            small = r
        if small == i:
            break
        heap[i], heap[small] = heap[small], heap[i]
        i = small
    return heap


sift_lesson()

[1, 3, 2, 7, 4]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. push(0): place it at the end
2. Parent is bigger → swap up to index 2
3. Parent is bigger → swap up to index 0
4. pop() returns 0; the last value moves to the root
5. A child is smaller → swap down to index 2

</details>


## Lesson 3: Using heapq in Python

*Level: Core*

```python
import heapq
h = []
heapq.heappush(h, 5)
heapq.heappush(h, 1)
heapq.heappop(h)          # 1 — always the smallest
h[0]                      # peek the smallest without removing
heapq.heapify(nums)       # turn a list into a heap in O(n)
heapq.nlargest(3, nums)   # convenience helpers
```



#### Max-heap



`heapq` is min-only. For a max-heap push `-x` and negate again when popping.



#### Custom priority



Push tuples: `(priority, tiebreak, item)`. Tuples compare element by element, so the first field decides. Add an index as tiebreak when items themselves can't be compared (like nodes).

**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q17](#q17), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 4: Top-k with a heap of size k

*Level: Core*

"Find the k largest" — sorting is O(n log n). Better: keep a **min**-heap of at most k items. For each new item push it; if the heap grows past k, pop the smallest. What remains are the k largest, and `heap[0]` is the k-th largest.



```python
for x in nums:
    heapq.heappush(h, x)
    if len(h) > k:
        heapq.heappop(h)
```



Cost O(n log k), memory O(k) — great when k is small or data is a stream. It feels backwards (min-heap for largest), so explain it in interviews: "the heap's top is the weakest of my current top k, so it's the one to kick out".



For "k closest" use a max-heap of size k on distance (push negative distance).

**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q1](#q1), [Q4](#q4), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 5: Merging k sorted things

*Level: Core*

When you have k sorted lists (or rows of a matrix, or a sorted grid of pair sums), the next smallest overall item is always one of the k current heads. Keep those heads in a heap:




1. Push the first item of every list (with its list index and position).
2. Pop the smallest, output it, and push the next item from the same list.



The heap never holds more than k items, so each step is O(log k). Used in merge k lists, k-th smallest in a sorted matrix, k pairs with smallest sums, and smallest range covering k lists (also track the current max).

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q9](#q9), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 6: Two heaps for a running median

*Level: Advanced*

Keep the smaller half of the numbers in a **max-heap** (`low`) and the larger half in a **min-heap** (`high`), with sizes equal or `low` one bigger. Then:




- the median is `low`'s top (odd count) or the average of both tops (even count) — O(1);
- adding a number: push into `low`, move `low`'s top to `high`, and if `high` becomes bigger move its top back — O(log n).



The two-heap idea also appears in sliding-window median and in "IPO"-style problems with one heap of locked items and one of available items.

**✏️ Your turn:** you now have what you need for [Q18](#q18), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 7: Greedy scheduling with a heap

*Level: Advanced*

A very common hard-problem pattern: **sort by time**, then sweep, and use a heap to always pick the best option *currently available*.




- **Meeting rooms**: sort by start; a min-heap of end times tells you whether the earliest-finishing room is free.
- **Single-threaded CPU**: sort tasks by arrival; push every task that has arrived; run the shortest.
- **Task scheduler**: max-heap of remaining counts plus a cooldown queue.
- **Furthest building**: push every climb; when climbs exceed ladders, pay for the smallest climb with bricks.
- **Maximum subsequence score**: sort by the multiplier descending; keep the k largest sums in a min-heap.

**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q12](#q12), [Q6](#q6), [Q13](#q13), [Q16](#q16), [Q14](#q14), [Q7](#q7), [Q8](#q8). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Kth Largest Element in a Stream

🟢 Warm-up · *Min-heap of size k · design*

Design a class that, after each `add(val)`, returns the k-th largest value seen so far.

**Examples**

- `KthLargest(3, [4, 5, 8, 2]) → add(3) → add(5) → add(10) → add(9) → add(4)` → `[4, 5, 5, 8, 8]`

In [ ]:
class KthLargest:
    """
    Design a class that, after each add(val), returns the k-th largest value
    seen so far.
    """

    def __init__(self, k, nums):
        pass

    def add(self, val):
        pass


# ▶ Run this cell to test your answer
check(KthLargest, [
    (((3, [4, 5, 8, 2]), [['add', (3,)], ['add', (5,)], ['add', (10,)], ['add', (9,)], ['add', (4,)]]), [4, 5, 5, 8, 8]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep every value in a sorted list (insort) and index from the end.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep only the k largest values in a min-heap; its top is the k-th largest.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Keep every value in a sorted list (insort) and index from the end.

```python
class KthLargest:
    def __init__(self, k, nums):
        self.k, self.vals = k, sorted(nums)

    def add(self, val):
        from bisect import insort
        insort(self.vals, val)
        return self.vals[-self.k]
```

**Cost:** add O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep only the k largest values in a min-heap; its top is the k-th largest.

```python
class KthLargest:
    def __init__(self, k, nums):
        import heapq
        self.k, self.heap = k, list(nums)
        heapq.heapify(self.heap)
        while len(self.heap) > k:
            heapq.heappop(self.heap)

    def add(self, val):
        import heapq
        heapq.heappush(self.heap, val)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)
        return self.heap[0]
```

**Cost:** add O(log k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep every value in a sorted list (insort) and index from the end. That costs add O(n) time and O(n) space."

"I can do better: Keep only the k largest values in a min-heap; its top is the k-th largest. That's add O(log k) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `KthLargest(3, [4, 5, 8, 2]) → add(3) → add(5) → add(10) → add(9) → add(4)`

1. add(3) → k-th largest = 4
2. add(5) → k-th largest = 5
3. add(10) → k-th largest = 5
4. add(9) → k-th largest = 8
5. add(4) → k-th largest = 8

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/kth-largest-element-in-a-stream)

<a id="q2"></a>
### Q2. Last Stone Weight

🟢 Warm-up · *Max-heap*

Repeatedly smash the two heaviest stones: equal → both vanish, else the difference remains. Return the last stone's weight or 0.

**Examples**

- `stones = [2, 7, 4, 1, 8, 1]` → `1`
- `stones = [1]` → `1`

In [ ]:
def last_stone_weight(stones):
    """
    Repeatedly smash the two heaviest stones: equal → both vanish, else the
    difference remains. Return the last stone's weight or 0.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(last_stone_weight, [
    (([2, 7, 4, 1, 8, 1],), 1),
    (([1],), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort the stones every round and take the last two.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Max-heap (negated values): pop two, push the difference if non-zero.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Sort the stones every round and take the last two.

```python
def last_stone_weight(stones):
    stones = list(stones)
    while len(stones) > 1:
        stones.sort()
        y, x = stones.pop(), stones.pop()
        if y != x:
            stones.append(y - x)
    return stones[0] if stones else 0
```

**Cost:** O(n² log n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Max-heap (negated values): pop two, push the difference if non-zero.

```python
def last_stone_weight(stones):
    import heapq
    heap = [-s for s in stones]
    heapq.heapify(heap)
    while len(heap) > 1:
        y, x = -heapq.heappop(heap), -heapq.heappop(heap)
        if y != x:
            heapq.heappush(heap, -(y - x))
    return -heap[0] if heap else 0
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort the stones every round and take the last two. That costs O(n² log n) time and O(1) space."

"I can do better: Max-heap (negated values): pop two, push the difference if non-zero. That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `stones = [2, 7, 4, 1, 8, 1]`

1. Smash 8 and 7 → leftover 1
2. Smash 4 and 2 → leftover 2
3. Smash 2 and 1 → leftover 1
4. Smash 1 and 1 → both gone

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/last-stone-weight)

<a id="q3"></a>
### Q3. Kth Largest Element in an Array

🟡 Interview standard · *Min-heap of size k*

Return the k-th largest element of `nums` (in sorted order, not the k-th distinct).

**Examples**

- `nums = [3, 2, 1, 5, 6, 4], k = 2` → `5`
- `nums = [3, 2, 3, 1, 2, 4, 5, 5, 6], k = 4` → `4`

In [ ]:
def find_kth_largest(nums, k):
    """
    Return the k-th largest element of nums (in sorted order, not the k-th
    distinct).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_kth_largest, [
    (([3, 2, 1, 5, 6, 4], 2), 5),
    (([3, 2, 3, 1, 2, 4, 5, 5, 6], 4), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort descending and index k − 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep a min-heap of the k largest seen. When it grows past k, pop the smallest. The top is the answer. (Quickselect gives O(n) average.)

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Sort descending and index k − 1.

```python
def find_kth_largest(nums, k):
    return sorted(nums, reverse=True)[k - 1]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep a min-heap of the k largest seen. When it grows past k, pop the smallest. The top is the answer. (Quickselect gives O(n) average.)

```python
def find_kth_largest(nums, k):
    import heapq
    heap = []
    for i, x in enumerate(nums):
        heapq.heappush(heap, x)
        if len(heap) > k:
            heapq.heappop(heap)
    return heap[0]
```

**Cost:** O(n log k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort descending and index k − 1. That costs O(n log n) time and O(n) space."

"I can do better: Keep a min-heap of the k largest seen. When it grows past k, pop the smallest. The top is the answer. (Quickselect gives O(n) average.) That's O(n log k) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [3, 2, 1, 5, 6, 4], k = 2`

1. Add 3; heap keeps the 2 largest, top = 3
2. Add 2; heap keeps the 2 largest, top = 2
3. Add 1; heap keeps the 2 largest, top = 2
4. Add 5; heap keeps the 2 largest, top = 3
5. Add 6; heap keeps the 2 largest, top = 5
6. Add 4; heap keeps the 2 largest, top = 5

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/kth-largest-element-in-an-array)

<a id="q4"></a>
### Q4. K Closest Points to Origin

🟡 Interview standard · *Max-heap of size k*

Return the `k` points closest to (0, 0) by Euclidean distance, in any order.

**Examples**

- `points = [[1, 3], [-2, 2]], k = 1` → `[[-2, 2]]`
- `points = [[3, 3], [5, -1], [-2, 4]], k = 2` → `[[3, 3], [-2, 4]]`

In [ ]:
def k_closest(points, k):
    """
    Return the k points closest to (0, 0) by Euclidean distance, in any order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(k_closest, [(([[1, 3], [-2, 2]], 1), [[-2, 2]]), (([[3, 3], [5, -1], [-2, 4]], 2), [[3, 3], [-2, 4]])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort all points by x² + y² and take k.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep a max-heap (negated distance) of the k closest so far; a closer point replaces the farthest.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Sort all points by x² + y² and take k.

```python
def k_closest(points, k):
    return sorted(points, key=lambda p: p[0] ** 2 + p[1] ** 2)[:k]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep a max-heap (negated distance) of the k closest so far; a closer point replaces the farthest.

```python
def k_closest(points, k):
    import heapq
    heap = []
    for x, y in points:
        d = x * x + y * y
        heapq.heappush(heap, (-d, x, y))
        if len(heap) > k:
            heapq.heappop(heap)
    return [[x, y] for _, x, y in heap]
```

**Cost:** O(n log k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort all points by x² + y² and take k. That costs O(n log n) time and O(n) space."

"I can do better: Keep a max-heap (negated distance) of the k closest so far; a closer point replaces the farthest. That's O(n log k) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `points = [[3, 3], [5, -1], [-2, 4], [1, 1], [0, 2]], k = 2`

1. Point (3,3) at distance² 18; keep the 2 closest
2. Point (5,-1) at distance² 26; keep the 2 closest
3. Point (-2,4) at distance² 20; keep the 2 closest
4. Point (1,1) at distance² 2; keep the 2 closest
5. Point (0,2) at distance² 4; keep the 2 closest

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/k-closest-points-to-origin)

<a id="q5"></a>
### Q5. Top K Frequent Words

🟡 Interview standard · *Heap with (−count, word)*

Return the `k` most frequent words, sorted by frequency then alphabetically.

**Examples**

- `words = ['i', 'love', 'leetcode', 'i', 'love', 'coding'], k = 2` → `['i', 'love']`
- `words = ['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'], k = 4` → `['the', 'is', 'sunny', 'day']`

In [ ]:
def top_k_frequent_words(words, k):
    """
    Return the k most frequent words, sorted by frequency then alphabetically.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_k_frequent_words, [
    ((['i', 'love', 'leetcode', 'i', 'love', 'coding'], 2), ['i', 'love']),
    ((['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'], 4), ['the', 'is', 'sunny', 'day']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count, then sort all distinct words by (−count, word).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count, heapify (−count, word) pairs in O(n), then pop k times. Ties come out alphabetically because words compare as strings.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count, then sort all distinct words by (−count, word).

```python
def top_k_frequent_words(words, k):
    count = {}
    for w in words:
        count[w] = count.get(w, 0) + 1
    return sorted(count, key=lambda w: (-count[w], w))[:k]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Count, heapify (−count, word) pairs in O(n), then pop k times. Ties come out alphabetically because words compare as strings.

```python
def top_k_frequent_words(words, k):
    import heapq
    count = {}
    for w in words:
        count[w] = count.get(w, 0) + 1
    heap = [(-c, w) for w, c in count.items()]
    heapq.heapify(heap)
    res = []
    for _ in range(k):
        c, w = heapq.heappop(heap)
        res.append(w)
    return res
```

**Cost:** O(n + k log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count, then sort all distinct words by (−count, word). That costs O(n log n) time and O(n) space."

"I can do better: Count, heapify (−count, word) pairs in O(n), then pop k times. Ties come out alphabetically because words compare as strings. That's O(n + k log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `words = ['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'], k = 4`

1. Pop 'the' (4 times)
2. Pop 'is' (3 times)
3. Pop 'sunny' (2 times)
4. Pop 'day' (1 times)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/top-k-frequent-words)

<a id="q6"></a>
### Q6. Task Scheduler

🟡 Interview standard · *Max-heap + cooldown queue*

Each CPU slot runs one task or idles; the same task needs `n` slots between runs. Return the fewest slots to finish all tasks.

**Examples**

- `tasks = ['A', 'A', 'A', 'B', 'B', 'B'], n = 2` → `8`
- `tasks = ['A', 'C', 'A', 'B', 'D', 'B'], n = 1` → `6`
- `tasks = ['A', 'A', 'A', 'B', 'B', 'B'], n = 3` → `10`

In [ ]:
def least_interval(tasks, n):
    """
    Each CPU slot runs one task or idles; the same task needs n slots between
    runs. Return the fewest slots to finish all tasks.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(least_interval, [
    ((['A', 'A', 'A', 'B', 'B', 'B'], 2), 8),
    ((['A', 'C', 'A', 'B', 'D', 'B'], 1), 6),
    ((['A', 'A', 'A', 'B', 'B', 'B'], 3), 10),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate each slot, scanning all task types to pick the available one with most remaining.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Simulate with a max-heap of remaining counts and a queue of tasks cooling down (count, ready time). Pop the most frequent ready task each slot. (A closed formula, (maxCount − 1)(n + 1) + #tasksWithMax, also works.)

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Simulate each slot, scanning all task types to pick the available one with most remaining.

```python
def least_interval(tasks, n):
    count = {}
    for t in tasks:
        count[t] = count.get(t, 0) + 1
    ready_at = {t: 0 for t in count}
    time = 0
    while count:
        avail = [t for t in count if ready_at[t] <= time]
        if avail:
            t = max(avail, key=lambda x: count[x])
            count[t] -= 1
            ready_at[t] = time + n + 1
            if count[t] == 0:
                del count[t]
        time += 1
    return time
```

**Cost:** O(T · 26) time, O(26) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Simulate with a max-heap of remaining counts and a queue of tasks cooling down (count, ready time). Pop the most frequent ready task each slot. (A closed formula, (maxCount − 1)(n + 1) + #tasksWithMax, also works.)

```python
def least_interval(tasks, n):
    import heapq
    from collections import Counter, deque
    heap = [-c for c in Counter(tasks).values()]
    heapq.heapify(heap)
    cooling, time = deque(), 0
    while heap or cooling:
        time += 1
        if heap:
            c = heapq.heappop(heap) + 1
            if c:
                cooling.append((c, time + n))
        if cooling and cooling[0][1] == time:
            heapq.heappush(heap, cooling.popleft()[0])
    return time
```

**Cost:** O(T · log 26) time, O(26) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate each slot, scanning all task types to pick the available one with most remaining. That costs O(T · 26) time and O(26) space."

"I can do better: Simulate with a max-heap of remaining counts and a queue of tasks cooling down (count, ready time). Pop the most frequent ready task each slot. (A closed formula, (maxCount − 1)(n + 1) + #tasksWithMax, also works.) That's O(T · log 26) time and O(26) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `tasks = ['A', 'A', 'A', 'B', 'B', 'C'], n = 2`

1. Slot 1
2. Slot 2
3. Slot 3
4. Slot 4
5. Slot 5
6. Slot 6
7. Slot 7

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/task-scheduler)

<a id="q7"></a>
### Q7. Reorganize String

🟡 Interview standard · *Max-heap, alternate the top two*

Rearrange `s` so no two adjacent characters are equal; return any such string or `""`.

**Examples**

- `s = 'aab'` → `'aba'`
- `s = 'aaab'` → `''`
- `s = 'vvvlo'` → `'vlvov'`

In [ ]:
def reorganize_string(s):
    """
    Rearrange s so no two adjacent characters are equal; return any such string
    or "".
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reorganize_string, [(("aab",), "aba"), (("aaab",), ""), (("vvvlo",), "vlvov")], verify=lambda a, out: (out == "" and max(a[0].count(c) for c in a[0]) > (len(a[0]) + 1) // 2) or (sorted(out) == sorted(a[0]) and all(x != y for x, y in zip(out, out[1:]))))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try permutations until one has no equal neighbours.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Always place the most frequent remaining letter that differs from the last one: pop the top; if it equals the last placed, pop the next one instead and push the first back.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try permutations until one has no equal neighbours.

```python
def reorganize_string(s):
    from itertools import permutations
    for p in set(permutations(s)):
        if all(a != b for a, b in zip(p, p[1:])):
            return "".join(p)
    return ""
```

**Cost:** O(n! · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Always place the most frequent remaining letter that differs from the last one: pop the top; if it equals the last placed, pop the next one instead and push the first back.

```python
def reorganize_string(s):
    import heapq
    from collections import Counter
    heap = [(-c, ch) for ch, c in Counter(s).items()]
    heapq.heapify(heap)
    res = []
    while heap:
        c, ch = heapq.heappop(heap)
        if res and res[-1] == ch:
            if not heap:
                return ""
            c2, ch2 = heapq.heappop(heap)
            res.append(ch2)
            if c2 + 1:
                heapq.heappush(heap, (c2 + 1, ch2))
            heapq.heappush(heap, (c, ch))
        else:
            res.append(ch)
            if c + 1:
                heapq.heappush(heap, (c + 1, ch))
    return "".join(res)
```

**Cost:** O(n log σ) time, O(σ) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try permutations until one has no equal neighbours. That costs O(n! · n) time and O(n) space."

"I can do better: Always place the most frequent remaining letter that differs from the last one: pop the top; if it equals the last placed, pop the next one instead and push the first back. That's O(n log σ) time and O(σ) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'aab'`

1. Place 'a'
2. Place 'b'
3. Place 'a'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/reorganize-string)

<a id="q8"></a>
### Q8. Sort Characters By Frequency

🟡 Interview standard · *Max-heap of counts*

Sort the characters of `s` by decreasing frequency (equal letters stay together).

**Examples**

- `s = 'tree'` → `'eert'`
- `s = 'cccaaa'` → `'aaaccc'`
- `s = 'Aabb'` → `'bbAa'`

In [ ]:
def _grouped_desc(out):
    runs = []
    for ch in out:
        if runs and runs[-1][0] == ch:
            runs[-1][1] += 1
        else:
            runs.append([ch, 1])
    chars = [r[0] for r in runs]
    return len(chars) == len(set(chars)) and all(runs[i][1] >= runs[i + 1][1] for i in range(len(runs) - 1))

def frequency_sort(s):
    """
    Sort the characters of s by decreasing frequency (equal letters stay
    together).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(frequency_sort, [(("tree",), "eert"), (("cccaaa",), "aaaccc"), (("Aabb",), "bbAa")], verify=lambda a, out: sorted(out) == sorted(a[0]) and _grouped_desc(out))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count, then sort the characters with key (−count, char).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count, push (−count, char) into a heap, pop each and write it count times. (Bucket sort by count gives O(n).)

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count, then sort the characters with key (−count, char).

```python
def frequency_sort(s):
    count = {}
    for ch in s:
        count[ch] = count.get(ch, 0) + 1
    return "".join(sorted(s, key=lambda ch: (-count[ch], ch)))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Count, push (−count, char) into a heap, pop each and write it count times. (Bucket sort by count gives O(n).)

```python
def frequency_sort(s):
    import heapq
    from collections import Counter
    heap = [(-c, ch) for ch, c in Counter(s).items()]
    heapq.heapify(heap)
    out = []
    while heap:
        c, ch = heapq.heappop(heap)
        out.append(ch * -c)
    return "".join(out)
```

**Cost:** O(n + σ log σ) time, O(σ) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count, then sort the characters with key (−count, char). That costs O(n log n) time and O(n) space."

"I can do better: Count, push (−count, char) into a heap, pop each and write it count times. (Bucket sort by count gives O(n).) That's O(n + σ log σ) time and O(σ) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'tree'`

1. Write 'e' × 2
2. Write 'r' × 1
3. Write 't' × 1

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/sort-characters-by-frequency)

<a id="q9"></a>
### Q9. Find K Pairs with Smallest Sums

🟡 Interview standard · *Heap frontier on a sorted grid*

Given two sorted arrays, return the `k` pairs (u, v) with the smallest sums.

**Examples**

- `nums1 = [1, 7, 11], nums2 = [2, 4, 6], k = 3` → `[[1, 2], [1, 4], [1, 6]]`
- `nums1 = [1, 1, 2], nums2 = [1, 2, 3], k = 2` → `[[1, 1], [1, 1]]`

In [ ]:
def k_smallest_pairs(nums1, nums2, k):
    """
    Given two sorted arrays, return the k pairs (u, v) with the smallest sums.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(k_smallest_pairs, [(([1, 7, 11], [2, 4, 6], 3), [[1, 2], [1, 4], [1, 6]]), (([1, 1, 2], [1, 2, 3], 2), [[1, 1], [1, 1]])], norm=lambda r: sorted(r))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Generate all m·n pairs and sort by sum.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Think of an m × n grid of sums sorted along rows and columns. Seed the heap with (nums1[i] + nums2[0]) for each row, and after popping (i, j) push (i, j + 1).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Generate all m·n pairs and sort by sum.

```python
def k_smallest_pairs(nums1, nums2, k):
    pairs = [[a, b] for a in nums1 for b in nums2]
    pairs.sort(key=lambda p: p[0] + p[1])
    return pairs[:k]
```

**Cost:** O(mn log(mn)) time, O(mn) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Think of an m × n grid of sums sorted along rows and columns. Seed the heap with (nums1[i] + nums2[0]) for each row, and after popping (i, j) push (i, j + 1).

```python
def k_smallest_pairs(nums1, nums2, k):
    import heapq
    heap = [(nums1[i] + nums2[0], i, 0) for i in range(min(k, len(nums1)))]
    heapq.heapify(heap)
    res = []
    while heap and len(res) < k:
        s, i, j = heapq.heappop(heap)
        res.append([nums1[i], nums2[j]])
        if j + 1 < len(nums2):
            heapq.heappush(heap, (nums1[i] + nums2[j + 1], i, j + 1))
    return res
```

**Cost:** O(k log min(k, m)) time, O(min(k, m)) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Generate all m·n pairs and sort by sum. That costs O(mn log(mn)) time and O(mn) space."

"I can do better: Think of an m × n grid of sums sorted along rows and columns. Seed the heap with (nums1[i] + nums2[0]) for each row, and after popping (i, j) push (i, j + 1). That's O(k log min(k, m)) time and O(min(k, m)) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums1 = [1, 7, 11], nums2 = [2, 4, 6], k = 3`

1. Pop pair (1, 2) sum 3; push its right neighbour
2. Pop pair (1, 4) sum 5; push its right neighbour
3. Pop pair (1, 6) sum 7; push its right neighbour

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/find-k-pairs-with-smallest-sums)

<a id="q10"></a>
### Q10. Kth Smallest Element in a Sorted Matrix

🟡 Interview standard · *Heap of row heads*

Every row and column of an n × n matrix is sorted. Return the k-th smallest element.

**Examples**

- `matrix = [[1, 5, 9], [10, 11, 13], [12, 13, 15]], k = 8` → `13`
- `matrix = [[-5]], k = 1` → `-5`

In [ ]:
def kth_smallest_matrix(matrix, k):
    """
    Every row and column of an n × n matrix is sorted. Return the k-th smallest
    element.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(kth_smallest_matrix, [
    (([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8), 13),
    (([[-5]], 1), -5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Flatten and sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Push the first element of each row. Pop k − 1 times, each time pushing the next element in the popped row. The top is the answer. (Binary search on value also works.)

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Flatten and sort.

```python
def kth_smallest_matrix(matrix, k):
    return sorted(x for row in matrix for x in row)[k - 1]
```

**Cost:** O(n² log n) time, O(n²) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Push the first element of each row. Pop k − 1 times, each time pushing the next element in the popped row. The top is the answer. (Binary search on value also works.)

```python
def kth_smallest_matrix(matrix, k):
    import heapq
    heap = [(row[0], r, 0) for r, row in enumerate(matrix)]
    heapq.heapify(heap)
    popped = []
    for _ in range(k - 1):
        v, r, c = heapq.heappop(heap)
        popped.append((r, c))
        if c + 1 < len(matrix[0]):
            heapq.heappush(heap, (matrix[r][c + 1], r, c + 1))
    return heap[0][0]
```

**Cost:** O(k log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Flatten and sort. That costs O(n² log n) time and O(n²) space."

"I can do better: Push the first element of each row. Pop k − 1 times, each time pushing the next element in the popped row. The top is the answer. (Binary search on value also works.) That's O(k log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `matrix = [[1, 5, 9], [10, 11, 13], [12, 13, 15]], k = 8`

1. Pop 1 (#1 smallest)
2. Pop 5 (#2 smallest)
3. Pop 9 (#3 smallest)
4. Pop 10 (#4 smallest)
5. Pop 11 (#5 smallest)
6. Pop 12 (#6 smallest)
7. Pop 13 (#7 smallest)
8. The heap top 13 is the 8-th smallest

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/kth-smallest-element-in-a-sorted-matrix)

<a id="q11"></a>
### Q11. Ugly Number II

🟡 Interview standard · *Heap of multiples*

Ugly numbers have no prime factors other than 2, 3 and 5 (1 counts). Return the n-th one.

**Examples**

- `n = 10` → `12`
- `n = 1` → `1`
- `n = 15` → `24`

In [ ]:
def nth_ugly_number(n):
    """
    Ugly numbers have no prime factors other than 2, 3 and 5 (1 counts). Return
    the n-th one.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(nth_ugly_number, [
    ((10,), 12),
    ((1,), 1),
    ((15,), 24),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Test 1, 2, 3, … dividing out 2, 3 and 5 until n ugly numbers are found.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Pop the smallest ugly number x from a heap and push 2x, 3x, 5x (skipping ones already seen). The n-th pop is the answer. (Three pointers give O(n).)

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Test 1, 2, 3, … dividing out 2, 3 and 5 until n ugly numbers are found.

```python
def nth_ugly_number(n):
    x, found = 0, 0
    while found < n:
        x += 1
        y = x
        for p in (2, 3, 5):
            while y % p == 0:
                y //= p
        found += y == 1
    return x
```

**Cost:** O(answer · log) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Pop the smallest ugly number x from a heap and push 2x, 3x, 5x (skipping ones already seen). The n-th pop is the answer. (Three pointers give O(n).)

```python
def nth_ugly_number(n):
    import heapq
    heap, seen = [1], {1}
    for i in range(n):
        x = heapq.heappop(heap)
        for p in (2, 3, 5):
            if x * p not in seen:
                seen.add(x * p)
                heapq.heappush(heap, x * p)
    return x
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Test 1, 2, 3, … dividing out 2, 3 and 5 until n ugly numbers are found. That costs O(answer · log) time and O(1) space."

"I can do better: Pop the smallest ugly number x from a heap and push 2x, 3x, 5x (skipping ones already seen). The n-th pop is the answer. (Three pointers give O(n).) That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 10`

1. Ugly #1 = 1; push 2, 3, 5
2. Ugly #2 = 2; push 4, 6, 10
3. Ugly #3 = 3; push 6, 9, 15
4. Ugly #4 = 4; push 8, 12, 20
5. Ugly #5 = 5; push 10, 15, 25
6. Ugly #6 = 6; push 12, 18, 30
7. Ugly #7 = 8; push 16, 24, 40
8. Ugly #8 = 9; push 18, 27, 45
9. Ugly #9 = 10; push 20, 30, 50
10. Ugly #10 = 12; push 24, 36, 60

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/ugly-number-ii)

<a id="q12"></a>
### Q12. Single-Threaded CPU

🟡 Interview standard · *Sort by time + heap by (duration, index)*

Task i becomes available at `tasks[i][0]` and takes `tasks[i][1]`. An idle CPU picks the shortest available task (smallest index on ties). Return the order tasks run.

**Examples**

- `tasks = [[1, 2], [2, 4], [3, 2], [4, 1]]` → `[0, 2, 3, 1]`
- `tasks = [[7, 10], [7, 12], [7, 5], [7, 4], [7, 2]]` → `[4, 3, 2, 0, 1]`

In [ ]:
def get_order(tasks):
    """
    Task i becomes available at tasks[i][0] and takes tasks[i][1]. An idle CPU
    picks the shortest available task (smallest index on ties). Return the order
    tasks run.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(get_order, [
    (([[1, 2], [2, 4], [3, 2], [4, 1]],), [0, 2, 3, 1]),
    (([[7, 10], [7, 12], [7, 5], [7, 4], [7, 2]],), [4, 3, 2, 0, 1]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

At each step scan all tasks for available ones and pick the best.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort task indices by start time. Push every task that has arrived into a heap keyed by (duration, index); if none has arrived, jump time forward.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** At each step scan all tasks for available ones and pick the best.

```python
def get_order(tasks):
    done, order, time = [False] * len(tasks), [], 0
    while len(order) < len(tasks):
        avail = [i for i in range(len(tasks)) if not done[i] and tasks[i][0] <= time]
        if not avail:
            time = min(tasks[i][0] for i in range(len(tasks)) if not done[i])
            continue
        i = min(avail, key=lambda i: (tasks[i][1], i))
        done[i] = True
        order.append(i)
        time += tasks[i][1]
    return order
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort task indices by start time. Push every task that has arrived into a heap keyed by (duration, index); if none has arrived, jump time forward.

```python
def get_order(tasks):
    import heapq
    idx = sorted(range(len(tasks)), key=lambda i: tasks[i][0])
    heap, order, time, j = [], [], 0, 0
    while len(order) < len(tasks):
        if not heap and time < tasks[idx[j]][0]:
            time = tasks[idx[j]][0]
        while j < len(idx) and tasks[idx[j]][0] <= time:
            heapq.heappush(heap, (tasks[idx[j]][1], idx[j]))
            j += 1
        d, i = heapq.heappop(heap)
        time += d
        order.append(i)
    return order
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: At each step scan all tasks for available ones and pick the best. That costs O(n²) time and O(n) space."

"I can do better: Sort task indices by start time. Push every task that has arrived into a heap keyed by (duration, index); if none has arrived, jump time forward. That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `tasks = [[1, 2], [2, 4], [3, 2], [4, 1]]`

1. Run task 0 (duration 2); time is now 3
2. Run task 2 (duration 2); time is now 5
3. Run task 3 (duration 1); time is now 6
4. Run task 1 (duration 4); time is now 10

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/single-threaded-cpu)

<a id="q13"></a>
### Q13. Furthest Building You Can Reach

🟡 Interview standard · *Min-heap of ladder climbs*

Climbing up costs bricks equal to the height gain, or one ladder. Return the furthest building index you can reach.

**Examples**

- `heights = [4, 2, 7, 6, 9, 14, 12], bricks = 5, ladders = 1` → `4`
- `heights = [4, 12, 2, 7, 3, 18, 20, 3, 19], bricks = 10, ladders = 2` → `7`
- `heights = [14, 3, 19, 3], bricks = 17, ladders = 0` → `3`

In [ ]:
def furthest_building(heights, bricks, ladders):
    """
    Climbing up costs bricks equal to the height gain, or one ladder. Return the
    furthest building index you can reach.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(furthest_building, [
    (([4, 2, 7, 6, 9, 14, 12], 5, 1), 4),
    (([4, 12, 2, 7, 3, 18, 20, 3, 19], 10, 2), 7),
    (([14, 3, 19, 3], 17, 0), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try both choices (bricks or ladder) at every climb.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use ladders on the biggest climbs. Push each climb into a min-heap; once it holds more climbs than ladders, pay for the smallest one with bricks. Stop when bricks go negative.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try both choices (bricks or ladder) at every climb.

```python
def furthest_building(heights, bricks, ladders):
    def go(i, b, l):
        if i == len(heights) - 1:
            return i
        d = heights[i + 1] - heights[i]
        if d <= 0:
            return go(i + 1, b, l)
        best = i
        if b >= d:
            best = max(best, go(i + 1, b - d, l))
        if l:
            best = max(best, go(i + 1, b, l - 1))
        return best
    return go(0, bricks, ladders)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Use ladders on the biggest climbs. Push each climb into a min-heap; once it holds more climbs than ladders, pay for the smallest one with bricks. Stop when bricks go negative.

```python
def furthest_building(heights, bricks, ladders):
    import heapq
    heap = []
    for i in range(len(heights) - 1):
        d = heights[i + 1] - heights[i]
        if d <= 0:
            continue
        heapq.heappush(heap, d)
        if len(heap) > ladders:
            bricks -= heapq.heappop(heap)
        if bricks < 0:
            return i
    return len(heights) - 1
```

**Cost:** O(n log L) time, O(L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try both choices (bricks or ladder) at every climb. That costs O(2ⁿ) time and O(n) space."

"I can do better: Use ladders on the biggest climbs. Push each climb into a min-heap; once it holds more climbs than ladders, pay for the smallest one with bricks. Stop when bricks go negative. That's O(n log L) time and O(L) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `heights = [4, 2, 7, 6, 9, 14, 12], bricks = 5, ladders = 1`

1. Climb 5 to building 2; ladders cover the biggest climbs, bricks left 5
2. Climb 3 to building 4; ladders cover the biggest climbs, bricks left 2
3. Climb 5 to building 5; ladders cover the biggest climbs, bricks left -3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/furthest-building-you-can-reach)

<a id="q14"></a>
### Q14. Hand of Straights

🟡 Interview standard · *Count + min-heap of starts*

Can the cards in `hand` be split into groups of `groupSize` consecutive values?

**Examples**

- `hand = [1, 2, 3, 6, 2, 3, 4, 7, 8], group_size = 3` → `True`
- `hand = [1, 2, 3, 4, 5], group_size = 4` → `False`

In [ ]:
def is_n_straight_hand(hand, group_size):
    """
    Can the cards in hand be split into groups of groupSize consecutive values?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_n_straight_hand, [
    (([1, 2, 3, 6, 2, 3, 4, 7, 8], 3), True),
    (([1, 2, 3, 4, 5], 4), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort; repeatedly remove the smallest card and the next groupSize − 1 values from the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The smallest remaining card must start a group. Keep counts and a min-heap of values; take groupSize consecutive values from the smallest, decrementing counts.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Sort; repeatedly remove the smallest card and the next groupSize − 1 values from the list.

```python
def is_n_straight_hand(hand, group_size):
    cards = sorted(hand)
    while cards:
        start = cards[0]
        for v in range(start, start + group_size):
            if v not in cards:
                return False
            cards.remove(v)
    return True
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** The smallest remaining card must start a group. Keep counts and a min-heap of values; take groupSize consecutive values from the smallest, decrementing counts.

```python
def is_n_straight_hand(hand, group_size):
    import heapq
    from collections import Counter
    if len(hand) % group_size:
        return False
    count = Counter(hand)
    heap = list(count)
    heapq.heapify(heap)
    while heap:
        start = heap[0]
        for v in range(start, start + group_size):
            if count[v] == 0:
                return False
            count[v] -= 1
            if count[v] == 0:
                if v != heap[0]:
                    return False
                heapq.heappop(heap)
    return True
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort; repeatedly remove the smallest card and the next groupSize − 1 values from the list. That costs O(n²) time and O(n) space."

"I can do better: The smallest remaining card must start a group. Keep counts and a min-heap of values; take groupSize consecutive values from the smallest, decrementing counts. That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `hand = [1, 2, 3, 6, 2, 3, 4, 7, 8], group_size = 3`

1. Group [1, 2, 3] formed
2. Group [2, 3, 4] formed
3. Group [6, 7, 8] formed

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/hand-of-straights)

<a id="q15"></a>
### Q15. Meeting Rooms II

🟡 Interview standard · *Min-heap of end times*

Given meeting intervals, return the minimum number of rooms needed.

**Examples**

- `intervals = [[0, 30], [5, 10], [15, 20]]` → `2`
- `intervals = [[7, 10], [2, 4]]` → `1`
- `intervals = [[1, 5], [2, 6], [3, 7], [6, 8]]` → `3`

In [ ]:
def min_meeting_rooms(intervals):
    """
    Given meeting intervals, return the minimum number of rooms needed.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_meeting_rooms, [
    (([[0, 30], [5, 10], [15, 20]],), 2),
    (([[7, 10], [2, 4]],), 1),
    (([[1, 5], [2, 6], [3, 7], [6, 8]],), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every meeting start, count how many meetings are running at that moment.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort by start. A min-heap holds end times of rooms in use; if the earliest-ending room is free by this start, reuse it (pop), then push this meeting's end. The heap size is the answer.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every meeting start, count how many meetings are running at that moment.

```python
def min_meeting_rooms(intervals):
    best = 0
    for s, _ in intervals:
        best = max(best, sum(1 for a, b in intervals if a <= s < b))
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort by start. A min-heap holds end times of rooms in use; if the earliest-ending room is free by this start, reuse it (pop), then push this meeting's end. The heap size is the answer.

```python
def min_meeting_rooms(intervals):
    import heapq
    intervals = sorted(intervals)
    ends = []
    for s, e in intervals:
        if ends and ends[0] <= s:
            heapq.heappop(ends)
        heapq.heappush(ends, e)
    return len(ends)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every meeting start, count how many meetings are running at that moment. That costs O(n²) time and O(1) space."

"I can do better: Sort by start. A min-heap holds end times of rooms in use; if the earliest-ending room is free by this start, reuse it (pop), then push this meeting's end. The heap size is the answer. That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `intervals = [[0, 30], [5, 10], [15, 20]]`

1. Meeting [0, 30): rooms in use end at [30]
2. Meeting [5, 10): rooms in use end at [10, 30]
3. Meeting [15, 20): rooms in use end at [20, 30]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/meeting-rooms-ii)

<a id="q16"></a>
### Q16. Maximum Subsequence Score

🟡 Interview standard · *Sort by multiplier + min-heap of sums*

Choose k indices; score = (sum of their nums1) × (min of their nums2). Return the max score.

**Examples**

- `nums1 = [1, 3, 3, 2], nums2 = [2, 1, 3, 4], k = 3` → `12`
- `nums1 = [4, 2, 3, 1, 1], nums2 = [7, 5, 10, 9, 6], k = 1` → `30`

In [ ]:
def max_score_subseq(nums1, nums2, k):
    """
    Choose k indices; score = (sum of their nums1) × (min of their nums2).
    Return the max score.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_score_subseq, [
    (([1, 3, 3, 2], [2, 1, 3, 4], 3), 12),
    (([4, 2, 3, 1, 1], [7, 5, 10, 9, 6], 1), 30),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every combination of k indices.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort pairs by nums2 descending so the current nums2 is the minimum. Keep the k largest nums1 seen in a min-heap with their sum; score = sum × current nums2.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every combination of k indices.

```python
def max_score_subseq(nums1, nums2, k):
    from itertools import combinations
    return max(sum(nums1[i] for i in c) * min(nums2[i] for i in c) for c in combinations(range(len(nums1)), k))
```

**Cost:** O(C(n, k) · k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort pairs by nums2 descending so the current nums2 is the minimum. Keep the k largest nums1 seen in a min-heap with their sum; score = sum × current nums2.

```python
def max_score_subseq(nums1, nums2, k):
    import heapq
    pairs = sorted(zip(nums2, nums1), reverse=True)
    heap, total, best = [], 0, 0
    for m, x in pairs:
        heapq.heappush(heap, x)
        total += x
        if len(heap) > k:
            total -= heapq.heappop(heap)
        if len(heap) == k:
            best = max(best, total * m)
    return best
```

**Cost:** O(n log n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every combination of k indices. That costs O(C(n, k) · k) time and O(k) space."

"I can do better: Sort pairs by nums2 descending so the current nums2 is the minimum. Keep the k largest nums1 seen in a min-heap with their sum; score = sum × current nums2. That's O(n log n) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums1 = [1, 3, 3, 2], nums2 = [2, 1, 3, 4], k = 3`

1. min nums2 = 4; keep the 3 largest nums1 (sum 2)
2. min nums2 = 3; keep the 3 largest nums1 (sum 5)
3. min nums2 = 2; keep the 3 largest nums1 (sum 6)
4. min nums2 = 1; keep the 3 largest nums1 (sum 8)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/maximum-subsequence-score)

<a id="q17"></a>
### Q17. Remove Stones to Minimize the Total

🟡 Interview standard · *Max-heap*

`k` times, pick a pile and remove ⌊pile / 2⌋ stones from it. Return the minimum total left.

**Examples**

- `piles = [5, 4, 9], k = 2` → `12`
- `piles = [4, 3, 6, 7], k = 3` → `12`

In [ ]:
def min_stone_sum(piles, k):
    """
    k times, pick a pile and remove ⌊pile / 2⌋ stones from it. Return the
    minimum total left.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_stone_sum, [
    (([5, 4, 9], 2), 12),
    (([4, 3, 6, 7], 3), 12),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Each time, scan for the largest pile.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Always halve the largest pile: max-heap, pop, push back pile − pile // 2.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Each time, scan for the largest pile.

```python
def min_stone_sum(piles, k):
    piles = list(piles)
    for _ in range(k):
        i = piles.index(max(piles))
        piles[i] -= piles[i] // 2
    return sum(piles)
```

**Cost:** O(k · n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Always halve the largest pile: max-heap, pop, push back pile − pile // 2.

```python
def min_stone_sum(piles, k):
    import heapq
    heap = [-p for p in piles]
    heapq.heapify(heap)
    for _ in range(k):
        p = -heapq.heappop(heap)
        heapq.heappush(heap, -(p - p // 2))
    return -sum(heap)
```

**Cost:** O(n + k log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Each time, scan for the largest pile. That costs O(k · n) time and O(1) space."

"I can do better: Always halve the largest pile: max-heap, pop, push back pile − pile // 2. That's O(n + k log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `piles = [4, 3, 6, 7], k = 3`

1. Halve the biggest pile 7 → 4
2. Halve the biggest pile 6 → 3
3. Halve the biggest pile 4 → 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/remove-stones-to-minimize-the-total)

<a id="q18"></a>
### Q18. Find Median from Data Stream

🔴 Stretch · *Two heaps*

Design `addNum(num)` and `findMedian()` for a growing stream of numbers.

**Examples**

- `MedianFinder() → addNum(1) → addNum(2) → findMedian() → addNum(3) → findMedian()` → `[None, None, 1.5, None, 2.0]`

In [ ]:
class MedianFinder:
    """
    Design addNum(num) and findMedian() for a growing stream of numbers.
    """

    def __init__(self):
        pass

    def addNum(self, num):
        pass

    def findMedian(self):
        pass


# ▶ Run this cell to test your answer
check(MedianFinder, [
    (((), [['addNum', (1,)], ['addNum', (2,)], ['findMedian', ()], ['addNum', (3,)], ['findMedian', ()]]), [None, None, 1.5, None, 2.0]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a sorted list with insort; the median is the middle.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A max-heap 'low' holds the smaller half and a min-heap 'high' the larger half, balanced so low has the same size or one more. The median is low's top or the average of both tops.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Keep a sorted list with insort; the median is the middle.

```python
class MedianFinder:
    def __init__(self):
        self.vals = []

    def addNum(self, num):
        from bisect import insort
        insort(self.vals, num)

    def findMedian(self):
        n = len(self.vals)
        return float(self.vals[n // 2]) if n % 2 else (self.vals[n // 2 - 1] + self.vals[n // 2]) / 2
```

**Cost:** addNum O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** A max-heap 'low' holds the smaller half and a min-heap 'high' the larger half, balanced so low has the same size or one more. The median is low's top or the average of both tops.

```python
class MedianFinder:
    def __init__(self):
        self.low, self.high = [], []  # low is a max-heap (negated)

    def addNum(self, num):
        import heapq
        heapq.heappush(self.low, -num)
        heapq.heappush(self.high, -heapq.heappop(self.low))
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def findMedian(self):
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2
```

**Cost:** addNum O(log n), findMedian O(1) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a sorted list with insort; the median is the middle. That costs addNum O(n) time and O(n) space."

"I can do better: A max-heap 'low' holds the smaller half and a min-heap 'high' the larger half, balanced so low has the same size or one more. The median is low's top or the average of both tops. That's addNum O(log n), findMedian O(1) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `MedianFinder() → addNum(5) → addNum(15) → addNum(1) → findMedian() → addNum(3) → findMedian()`

1. addNum(5) and rebalance
2. addNum(15) and rebalance
3. addNum(1) and rebalance
4. addNum(3) and rebalance

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/find-median-from-data-stream)

<a id="q19"></a>
### Q19. IPO

🔴 Stretch · *Sort by capital + max-heap of profit*

You start with capital `w` and may finish at most `k` projects; a project needs `capital[i]` to start and adds `profits[i]`. Maximise final capital.

**Examples**

- `k = 2, w = 0, profits = [1, 2, 3], capital = [0, 1, 1]` → `4`
- `k = 3, w = 0, profits = [1, 2, 3], capital = [0, 1, 2]` → `6`

In [ ]:
def find_maximized_capital(k, w, profits, capital):
    """
    You start with capital w and may finish at most k projects; a project needs
    capital[i] to start and adds profits[i]. Maximise final capital.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_maximized_capital, [
    ((2, 0, [1, 2, 3], [0, 1, 1]), 4),
    ((3, 0, [1, 2, 3], [0, 1, 2]), 6),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Each round, scan every unused project you can afford and take the most profitable.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort projects by capital. Each round push every newly affordable project's profit into a max-heap, then take the top.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Each round, scan every unused project you can afford and take the most profitable.

```python
def find_maximized_capital(k, w, profits, capital):
    used = [False] * len(profits)
    for _ in range(k):
        best = -1
        for i in range(len(profits)):
            if not used[i] and capital[i] <= w and (best < 0 or profits[i] > profits[best]):
                best = i
        if best < 0:
            break
        used[best] = True
        w += profits[best]
    return w
```

**Cost:** O(k · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort projects by capital. Each round push every newly affordable project's profit into a max-heap, then take the top.

```python
def find_maximized_capital(k, w, profits, capital):
    import heapq
    projects = sorted(zip(capital, profits))
    heap, i = [], 0
    for _ in range(k):
        while i < len(projects) and projects[i][0] <= w:
            heapq.heappush(heap, -projects[i][1])
            i += 1
        if not heap:
            break
        gain = -heapq.heappop(heap)
        w += gain
    return w
```

**Cost:** O((n + k) log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Each round, scan every unused project you can afford and take the most profitable. That costs O(k · n) time and O(n) space."

"I can do better: Sort projects by capital. Each round push every newly affordable project's profit into a max-heap, then take the top. That's O((n + k) log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `k = 3, w = 0, profits = [1, 2, 3, 5], capital = [0, 1, 1, 3]`

1. Affordable profits in the heap; take 1 → capital 1
2. Affordable profits in the heap; take 3 → capital 4
3. Affordable profits in the heap; take 5 → capital 9

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/ipo)

<a id="q20"></a>
### Q20. Smallest Range Covering Elements from K Lists

🔴 Stretch · *Heap of one per list + running max*

Find the smallest range [a, b] that includes at least one number from each of the k sorted lists.

**Examples**

- `nums = [[4, 10, 15, 24, 26], [0, 9, 12, 20], [5, 18, 22, 30]]` → `[20, 24]`
- `nums = [[1, 2, 3], [1, 2, 3], [1, 2, 3]]` → `[1, 1]`

In [ ]:
def smallest_range(nums):
    """
    Find the smallest range [a, b] that includes at least one number from each
    of the k sorted lists.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(smallest_range, [
    (([[4, 10, 15, 24, 26], [0, 9, 12, 20], [5, 18, 22, 30]],), [20, 24]),
    (([[1, 2, 3], [1, 2, 3], [1, 2, 3]],), [1, 1]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every pair of values as the range ends and check that each list has a value inside.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Hold one element from every list in a min-heap and track the max. The range [heap min, max] covers all lists. Advance the list that gave the min; stop when a list runs out.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every pair of values as the range ends and check that each list has a value inside.

```python
def smallest_range(nums):
    from bisect import bisect_left
    vals = sorted({x for row in nums for x in row})
    best = None
    for a in vals:
        for b in vals:
            if b < a:
                continue
            if all(bisect_left(row, a) < len(row) and row[bisect_left(row, a)] <= b for row in nums):
                if best is None or (b - a, a) < (best[1] - best[0], best[0]):
                    best = [a, b]
                break
    return best
```

**Cost:** O(N² · k log) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Hold one element from every list in a min-heap and track the max. The range [heap min, max] covers all lists. Advance the list that gave the min; stop when a list runs out.

```python
def smallest_range(nums):
    import heapq
    heap = [(row[0], r, 0) for r, row in enumerate(nums)]
    heapq.heapify(heap)
    high = max(row[0] for row in nums)
    best = [heap[0][0], high]
    while True:
        low, r, c = heapq.heappop(heap)
        if high - low < best[1] - best[0]:
            best = [low, high]
        if c + 1 == len(nums[r]):
            return best
        nxt = nums[r][c + 1]
        high = max(high, nxt)
        heapq.heappush(heap, (nxt, r, c + 1))
```

**Cost:** O(N log k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every pair of values as the range ends and check that each list has a value inside. That costs O(N² · k log) time and O(1) space."

"I can do better: Hold one element from every list in a min-heap and track the max. The range [heap min, max] covers all lists. Advance the list that gave the min; stop when a list runs out. That's O(N log k) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [[4, 10, 15, 24, 26], [0, 9, 12, 20], [5, 18, 22, 30]]`

1. Current range [0, 5] covers every list; best [0, 5]
2. Current range [4, 9] covers every list; best [0, 5]
3. Current range [5, 10] covers every list; best [0, 5]
4. Current range [9, 18] covers every list; best [0, 5]
5. Current range [10, 18] covers every list; best [0, 5]
6. Current range [12, 18] covers every list; best [0, 5]
7. Current range [15, 20] covers every list; best [0, 5]
8. Current range [18, 24] covers every list; best [0, 5]
9. Current range [20, 24] covers every list; best [20, 24]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/heap/smallest-range-covering-elements-from-k-lists)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Is a heap sorted?</b></summary>

No. Only parent–child order is guaranteed, so you can read the min (or max) in O(1), but getting all elements in order requires popping them one by one (heap sort).

</details>

<details>
<summary><b>2. Why is building a heap O(n)?</b></summary>

Sift-down cost depends on a node's height. Half the nodes are leaves (no work), a quarter move at most one level, an eighth at most two… The sum of heights over all nodes is O(n).

</details>

<details>
<summary><b>3. How do you get a max-heap with heapq?</b></summary>

Push `-x` and negate when you pop. For tuples, negate the priority field.

</details>

<details>
<summary><b>4. Heap vs BST for a priority queue?</b></summary>

Both give O(log n) insert and delete-min. A heap is simpler, uses an array (better cache, less memory) and builds in O(n). A balanced BST also supports search, delete-any and sorted iteration in O(log n), which a heap doesn't.

</details>

<details>
<summary><b>5. How do you find the k largest elements efficiently?</b></summary>

Keep a min-heap of size k: push each element and pop when the size exceeds k. The heap holds the k largest; its root is the k-th largest. O(n log k) time, O(k) space. Quickselect gives O(n) average.

</details>

<details>
<summary><b>6. How does a running median work with heaps?</b></summary>

A max-heap stores the lower half, a min-heap the upper half, sizes kept within one. The median is one root or the average of both roots: O(log n) per insert, O(1) per query.

</details>

<details>
<summary><b>7. Is heap sort stable?</b></summary>

No. Swapping the root with the last element can reorder equal keys. It is O(n log n) in every case and in place.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: What a priority queue is
- [ ] I can explain: How a heap is stored
- [ ] I can explain: Using heapq in Python
- [ ] I can explain: Top-k with a heap of size k
- [ ] I can explain: Merging k sorted things
- [ ] I can explain: Two heaps for a running median
- [ ] I can explain: Greedy scheduling with a heap

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Maximum Subsequence Score
- [ ] [Q17](#q17) Remove Stones to Minimize the Total
- [ ] [Q18](#q18) Find Median from Data Stream
- [ ] [Q19](#q19) IPO
- [ ] [Q20](#q20) Smallest Range Covering Elements from K Lists

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.